# Validate the composite scorer on the gold set

Runs the full pipeline on the hand-rated mini gold set and checks Spearman / Pearson
correlation between the composite `creativity_score` and the human ratings.

Use `enable_stage2=False` to validate the Stage-1-only signal without any API keys.

In [ ]:
import json
from pathlib import Path

import numpy as np
from scipy.stats import pearsonr, spearmanr

from creativity_evaluators.pipeline import evaluate_batch
from creativity_evaluators.types import EvalConfig

gold = json.loads(Path('../tests/fixtures/sample_responses.json').read_text())['items']
len(gold)

In [ ]:
config = EvalConfig(enable_stage2=False)  # set True (with API keys) for the full stack
records = [{'prompt': g['prompt'], 'response': g['response'], 'category': g['category']} for g in gold]
results = evaluate_batch(records, config)

predicted = np.array([r.creativity_score for r in results])
human = np.array([g['human_score'] for g in gold])
predicted[:5], human[:5]

In [ ]:
rho, _ = spearmanr(predicted, human)
r, _ = pearsonr(predicted, human)
print(f'Spearman: {rho:.3f}  (acceptance bar >= 0.5)')
print(f'Pearson:  {r:.3f}')

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(human, predicted)
plt.xlabel('human creativity rating (1-5)')
plt.ylabel('composite creativity_score')
plt.title('Composite vs. human ratings')
plt.show()